
# Treatment Effect Analysis

This notebook evaluates the impact of email campaigns on customer revenue and purchase conversion using the Hillstrom randomized experiment.

- **Primary metric:** Revenue per randomized customer (`spend`)
- **Secondary metric:** Purchase conversion rate (`conversion`)
- **Diagnostic metric:** Visit rate (`visit`)
- **Control group:** No E-Mail

Detailed KPI calculations and treatment effect estimates are available in `sql/04_experiment_outcomes.sql`.



## 1. Observed Treatment Effects

The observed purchase conversion rates were 0.57% for Control, 1.25% for Men's Email, and 0.88% for Women's Email.

| Metric | Men's Email vs Control | Women's Email vs Control |
|---|---:|---:|
| Incremental Revenue per Customer | +$0.7698 | +$0.4244 |
| Relative Revenue Lift | +117.93% | +65.02% |
| Conversion Rate Lift | +0.68 pp | +0.31 pp |
| Visit Rate Lift | +7.66 pp | +4.52 pp |

Both email campaigns showed positive observed treatment effects, with Men's Email producing larger point estimates.

Although a 0.68 percentage-point increase may appear small, it represents an approximately 119% relative improvement over the control conversion rate of 0.57%.

Statistical inference is required to assess the uncertainty of these estimates.


## 2. Revenue Effect Analysis

Revenue per randomized customer (`spend`) is the primary metric.

Two-sided Welch's t-tests are used to compare mean spending between each email treatment and Control, allowing for unequal group variances.

**Hypotheses**

- H0: Mean spending is equal between treatment and control.
- H1: Mean spending differs between treatment and control.

**Significance level:** 0.05

Individual-level spending data are used to avoid intermediate rounding. Bootstrap confidence intervals will be calculated as an additional robustness check.

In [10]:
import pandas as pd
from scipy.stats import ttest_ind

# Load individual-level data
df = pd.read_csv("../data/raw/hillstrom.csv")

control_spend = df.loc[df["segment"] == "No E-Mail", "spend"]
men_spend = df.loc[df["segment"] == "Mens E-Mail", "spend"]
women_spend = df.loc[df["segment"] == "Womens E-Mail", "spend"]

# Welch's t-tests
result_men = ttest_ind(men_spend, control_spend, equal_var=False)
result_women = ttest_ind(women_spend, control_spend, equal_var=False)

# 95% confidence intervals
ci_men = result_men.confidence_interval(confidence_level=0.95)
ci_women = result_women.confidence_interval(confidence_level=0.95)

# Men's Email vs Control
print("Men's Email vs Control")
print(f"Incremental revenue: ${men_spend.mean() - control_spend.mean():.4f}")
print(f"T-statistic: {result_men.statistic:.4f}")
print(f"P-value: {result_men.pvalue:.6g}")
print(f"95% CI: (${ci_men.low:.4f}, ${ci_men.high:.4f})")

print()

# Women's Email vs Control
print("Women's Email vs Control")
print(f"Incremental revenue: ${women_spend.mean() - control_spend.mean():.4f}")
print(f"T-statistic: {result_women.statistic:.4f}")
print(f"P-value: {result_women.pvalue:.6g}")
print(f"95% CI: (${ci_women.low:.4f}, ${ci_women.high:.4f})")

Men's Email vs Control
Incremental revenue: $0.7698
T-statistic: 5.3001
P-value: 1.16381e-07
95% CI: ($0.4851, $1.0545)

Women's Email vs Control
Incremental revenue: $0.4244
T-statistic: 3.2564
P-value: 0.0011294
95% CI: ($0.1690, $0.6799)


### Revenue Hypothesis Test Results

| Comparison | Incremental Revenue | 95% CI | T-statistic | P-value |
|---|---:|---:|---:|---:|
| Men's Email vs Control | +$0.7698 | [$0.4851, $1.0545] | 5.3001 | 1.16381e-07 |
| Women's Email vs Control | +$0.4244 | [$0.1690, $0.6799] | 3.2564 | 0.0011294 |

**Interpretation:** Both email campaigns showed statistically significant increases in mean revenue per customer relative to Control (p < 0.05).

Both 95% confidence intervals exclude zero, supporting positive revenue effects. Men's Email produced a larger point estimate, but statistical superiority over Women's Email has not yet been established.

Given the substantial variability in individual spending, bootstrap confidence intervals will be used to further assess the robustness of these findings.

## 3. Conversion Rate Analysis

Two-sided two-proportion Z-tests are used to compare purchase conversion rates between each email treatment and Control.

**Hypotheses**

- H0: Treatment and control conversion rates are equal.
- H1: Treatment and control conversion rates differ.

**Significance level:** 0.05

In [11]:
from statsmodels.stats.proportion import proportions_ztest

# Men's Email vs Control
z_men, p_men = proportions_ztest(
    count=[267, 122],
    nobs=[21307, 21306],
    alternative="two-sided"
)

# Women's Email vs Control
z_women, p_women = proportions_ztest(
    count=[189, 122],
    nobs=[21387, 21306],
    alternative="two-sided"
)

print("Men's Email vs Control")
print(f"Z-statistic: {z_men:.4f}")
print(f"P-value: {p_men:.6g}")

print()

print("Women's Email vs Control")
print(f"Z-statistic: {z_women:.4f}")
print(f"P-value: {p_women:.6g}")

Men's Email vs Control
Z-statistic: 7.3851
P-value: 1.52322e-13

Women's Email vs Control
Z-statistic: 3.7796
P-value: 0.000157105


### Conversion Rate Hypothesis Test Results

| Comparison | Z-statistic | P-value |
|---|---:|---:|
| Men's Email vs Control | 7.3851 | 1.52322e-13 |
| Women's Email vs Control | 3.7796 | 0.000157105 |

**Interpretation:** Both email campaigns showed statistically significant increases in purchase conversion relative to Control (p < 0.05).

However, these results do not establish campaign profitability or demonstrate that Men's Email is statistically superior to Women's Email.

Confidence intervals for conversion rate differences and multiple-testing adjustments will be addressed in subsequent analyses.